# Microsoft Fabric & PySpark Practice

This notebook contains common PySpark operations for a Medallion architecture on Microsoft Fabric.

## 1. Reading from Bronze and Cleaning
Assume we have raw CSV files in the Bronze folder.

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit, lower

# Read from OneLake Bronze
bronze_df = spark.read.format("csv").option("header", "true").load("Files/Bronze/SalesRaw.csv")

# Cleaning logic (Silver layer)
silver_df = bronze_df.select(
    col("SaleID").cast("int"),
    col("CustomerID").cast("int"),
    col("Amount").cast("decimal(18,2)"),
    lower(col("Status")).alias("Status"),
    current_timestamp().alias("IngestionTime"),
    lit("SalesRaw.csv").alias("SourceFile")
).filter(col("SaleID").isNotNull())

silver_df.show(5)

## 2. Writing to Delta (Silver)
Practice writing as a Delta table with partitioning.

In [ ]:
# Write to Silver table
silver_df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("Status") \
    .saveAsTable("Sales_Silver")

## 3. Merge Operation (Upsert Logic)
Implementing SCD Type 1 logic using Delta Table Merge.

In [ ]:
from delta.tables import *

# Assume delta_table is our target in Silver
delta_table = DeltaTable.forName(spark, "Sales_Silver")

# New data to upsert
new_data = silver_df.limit(10) 

delta_table.alias("target").merge(
    new_data.alias("source"),
    "target.SaleID = source.SaleID"
).whenMatchedUpdateAll() \
.whenNotMatchedInsertAll() \
.execute()

## 4. Performance Optimization (V-Order)
In Fabric, V-Order is applied by default during writes, but you can also manually optimize.

In [ ]:
spark.sql("OPTIMIZE Sales_Silver")
spark.sql("VACUUM Sales_Silver RETAIN 168 HOURS") # Keep 7 days of history